[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/templates/56_adamw.ipynb)

# 🔴 Hard: AdamW (Decoupled Weight Decay)

**AdamW** (Loshchilov & Hutter 2019) を実装する。Adam + **decoupled weight decay**。Transformer 学習の modern default。

> 💡 **どこで使う**：Transformer 学習の事実上の標準。Adam の L2 を decoupled WD に置き換えた版で、HF Transformers のデフォルト。

<details>
<summary>📖 もっと詳しく</summary>

Loshchilov & Hutter 2019。`p ← p · (1 - lr·λ)` を Adam update の前に適用。

落とし穴: WD は通常 LayerNorm の重みや bias には適用しない（param groups で分離）。`weight_decay=0.01–0.1` が定番、cosine LR (#30) との組み合わせが Transformer レシピの基本。

</details>

### Decoupled の捻り
plain Adam + L2 正則化は `λp` を moment update の **前** に gradient に足すので、L2 項が `√v_hat` で割られてしまう — 各 weight で実効 decay rate が変わる。AdamW は Adam update の後で decay を **param に直接** 適用するので、全 weight で 実効 decay rate が同じになる：

$$p \leftarrow p \cdot (1 - \text{lr} \cdot \lambda)$$
$$p \leftarrow p - \text{lr} \cdot \hat{m} / (\sqrt{\hat{v}} + \varepsilon)$$

### Killer test (interview の決め台詞)
`weight_decay > 0` で **gradient = 0** でも、AdamW は param を縮める（decoupled WD は `g` に依らず `p` に作用）。plain Adam + L2 だとそのケースで何もしない。これが両者を区別する。


### Signature
```python
class MyAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01): ...
    def zero_grad(self): ...
    def step(self): ...
```

### Rules
- `torch.optim.AdamW` や `torch.optim.Adam` は **使わない**
- Decoupled WD: `p *= (1 - lr * weight_decay)` を param に適用（gradient ではない）
- その後 Adam: m, v EMA + bias correction、`p -= lr * m_hat / (sqrt(v_hat) + eps)`
- `m, v` を zeros で、step counter `t = 0` で初期化。`step()` の最初に `t += 1`
- `p.grad is None` の param は skip（WD も適用しない — `torch.optim.AdamW` の挙動に合わせる）
- `step()` を `@torch.no_grad()` で wrap

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        pass  # hyperparam 保存、m=[zeros], v=[zeros], t=0 で初期化
    
    def zero_grad(self):
        pass
    
    def step(self):
        pass  # t += 1; 各 (p, m, v) で p.grad is not None なら:
              #   decoupled WD: p *= (1 - lr*wd)
              #   m = β1*m + (1-β1)*g; v = β2*v + (1-β2)*g²
              #   m_hat = m / (1 - β1^t); v_hat = v / (1 - β2^t)
              #   p -= lr * m_hat / (sqrt(v_hat) + eps)

In [ ]:
import torch
torch.manual_seed(0)
p = torch.randn(5, requires_grad=True)
opt = MyAdamW([p], lr=0.01, weight_decay=0.1)
for step in range(3):
    loss = (p ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'step {step}: |p| = {p.norm().item():.4f}')

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("adamw")

---

**詰まったら？** 解答を見る：

[![Open Solution in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/56_adamw_solution.ipynb) または [GitHub で開く](https://github.com/alextfkd/TorchCode/blob/master/solutions/56_adamw_solution.ipynb)